# 09 - Asesores

Este notebook prueba las funcionalidades de asesores de la API de IOL.

## Funcionalidades:
- Consultar movimientos de clientes
- Obtener preguntas del test de inversor
- Calcular y guardar perfil de inversor
- Operar en nombre de clientes

**IMPORTANTE:** Estos endpoints requieren ROL DE ASESOR en IOL.
Si no tienes este rol, recibiras errores de permisos.

**ADVERTENCIA:** Las operaciones que modifican datos estan COMENTADAS.

**Nota:** Requiere credenciales validas de IOL en el archivo `.env`

## Configuracion Inicial

In [ ]:
import sys
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../..'))

from pyIol import (
    IOLClient, IOLAPIError,
    MovimientoCliente, MovimientosAsesor,
    OpcionRespuesta, PreguntaTestInversor, TestInversor, RespuestaTest,
    PerfilInversor, ResultadoOperacionAsesor
)
print("Librerias importadas correctamente")
print(f"Fecha y hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nNOTA: Estos endpoints requieren ROL DE ASESOR")

In [ ]:
load_dotenv('../../.env')
USERNAME = os.getenv('IOL_USERNAME', 'tu_usuario_iol')
PASSWORD = os.getenv('IOL_PASSWORD', 'tu_password_iol')

if USERNAME == "tu_usuario_iol":
    print("ADVERTENCIA: Configura las credenciales en .env")
else:
    print(f"Credenciales configuradas - Usuario: {USERNAME}")

In [ ]:
try:
    client = IOLClient(USERNAME, PASSWORD)
    print("Cliente IOL creado correctamente")
except Exception as e:
    print(f"Error al crear cliente: {e}")
    client = None

## 1. Consultar Movimientos de Clientes

In [ ]:
# Movimientos de clientes
if client:
    try:
        print("Obteniendo movimientos de clientes (ultimos 30 dias)...")
        
        fecha_hasta = datetime.now()
        fecha_desde = fecha_hasta - timedelta(days=30)
        
        movimientos = client.get_advisor_movements(
            fecha_desde=fecha_desde,
            fecha_hasta=fecha_hasta
        )
        
        print(f"\nTotal de movimientos: {movimientos.total_registros}")
        print(f"Pagina: {movimientos.numero_pagina}")
        
        if movimientos.movimientos:
            print("\nUltimos movimientos:")
            for mov in movimientos.movimientos[:5]:
                print(f"  [{mov.fecha_operada}] {mov.tipo_movimiento}")
                print(f"    Simbolo: {mov.simbolo} | Cantidad: {mov.cantidad}")
                print(f"    Monto: ${mov.monto:,.2f} {mov.moneda}")
        else:
            print("No hay movimientos en el periodo")
            
    except Exception as e:
        print(f"Error (probablemente requiere rol de asesor): {e}")

## 2. Test de Inversor

In [ ]:
# Obtener preguntas del test
if client:
    try:
        print("Obteniendo preguntas del test de inversor...")
        test = client.get_investor_test_questions()
        
        print(f"\nTotal de preguntas: {len(test.preguntas)}")
        
        for i, pregunta in enumerate(test.preguntas[:3], 1):
            print(f"\nPregunta {i} (ID: {pregunta.id}):")
            print(f"  {pregunta.descripcion}")
            print(f"  Opciones:")
            for opcion in pregunta.opciones:
                print(f"    [{opcion.id}] {opcion.descripcion}")
        
        if len(test.preguntas) > 3:
            print(f"\n  ... y {len(test.preguntas) - 3} preguntas mas")
            
    except Exception as e:
        print(f"Error (probablemente requiere rol de asesor): {e}")

In [ ]:
# Calcular perfil (sin guardar)
if client:
    try:
        print("Calculando perfil de inversor...")
        
        # Obtener preguntas primero
        test = client.get_investor_test_questions()
        
        # Crear respuestas de ejemplo (primera opcion de cada pregunta)
        respuestas = []
        for pregunta in test.preguntas:
            if pregunta.opciones:
                respuestas.append(RespuestaTest(
                    id_pregunta=pregunta.id,
                    id_respuesta=pregunta.opciones[0].id
                ))
        
        print(f"Enviando {len(respuestas)} respuestas...")
        
        perfil = client.calculate_investor_profile(respuestas)
        
        print(f"\nResultado:")
        print(f"  Perfil: {perfil.perfil}")
        print(f"  Descripcion: {perfil.descripcion}")
        print(f"  Puntaje: {perfil.puntaje}")
        print(f"  Guardado: {'Si' if perfil.guardado else 'No (solo calculo)'}")
        
    except Exception as e:
        print(f"Error (probablemente requiere rol de asesor): {e}")

## 3. Guardar Perfil de Inversor (COMENTADO)

In [ ]:
# GUARDAR PERFIL - COMENTADO POR SEGURIDAD
# Esta operacion MODIFICA datos del cliente

# if client:
#     try:
#         id_cliente = "123456"  # Reemplazar con ID real
#         
#         test = client.get_investor_test_questions()
#         respuestas = [
#             RespuestaTest(id_pregunta=p.id, id_respuesta=p.opciones[0].id)
#             for p in test.preguntas if p.opciones
#         ]
#         
#         perfil = client.save_investor_profile(
#             id_cliente=id_cliente,
#             respuestas=respuestas
#         )
#         
#         print(f"Perfil guardado: {perfil.perfil}")
#     except Exception as e:
#         print(f"Error: {e}")

print("Guardar perfil COMENTADO - Descomenta para ejecutar")

## 4. Operar como Asesor (COMENTADO)

In [ ]:
# OPERAR EN NOMBRE DE CLIENTE - COMENTADO POR SEGURIDAD
# Esta operacion ejecuta trading REAL

# if client:
#     try:
#         resultado = client.advisor_sell_dollar_bond(
#             id_cliente="123456",
#             simbolo="AL30D",
#             cantidad=1,
#             precio=50.5,
#             mercado="bCBA",
#             plazo="t1",
#             validez="2024-12-31"
#         )
#         
#         if resultado.ok:
#             print(f"Orden ejecutada: #{resultado.numero_operacion}")
#         else:
#             print(f"Error: {resultado.mensaje}")
#     except Exception as e:
#         print(f"Error: {e}")

print("Operar como asesor COMENTADO - Descomenta para ejecutar")

## Metodos RAW Disponibles

In [ ]:
print("METODOS RAW DE ASESORES")
print("="*50)
print("""
Para obtener respuestas en formato JSON crudo:

Consultas:
- client.get_advisor_movements_raw(fecha_desde, fecha_hasta, id_cliente)
- client.get_investor_test_questions_raw()
- client.calculate_investor_profile_raw(respuestas)

Operaciones:
- client.save_investor_profile_raw(id_cliente, respuestas)
- client.advisor_sell_dollar_bond_raw(id_cliente, simbolo, cantidad, precio, ...)

NOTA: Todos estos endpoints requieren ROL DE ASESOR.
""")

## Limpieza

In [ ]:
if client:
    try:
        client.close()
        print("Cliente IOL cerrado correctamente")
    except Exception as e:
        print(f"Error al cerrar cliente: {e}")